# JobMatch AI — Notebook 05: Sistema de Recomendação

**Projeto:** JobMatch AI — Sistema de Matching Currículo-Vaga com NLP/Deep Learning
**Autor:** Eduardo Matos
**Etapa:** Construção da função de predição — o "produto" final do projeto

## Objetivo

Transformar o modelo fine-tuned (Notebook 04) em uma ferramenta prática: uma
função que recebe o texto de uma vaga nova (que o modelo nunca viu, nem no
treino nem na validação) e retorna um fit_percentual estimado, junto com uma
interpretação legível do resultado.

Esta é a peça que conecta todo o trabalho anterior (coleta, EDA, tokenização,
embeddings, fine-tuning) a um caso de uso real e demonstrável — o tipo de
funcionalidade que pode ser mostrada rodando ao vivo numa entrevista técnica.

## O que este notebook faz
1. Carrega o modelo fine-tuned salvo (Notebook 04)
2. Constrói uma função `prever_fit(texto_vaga)` reutilizável
3. Testa a função em 2-3 vagas reais que ainda não estavam no dataset
4. Adiciona uma camada de interpretação (ex: "Fit alto — área técnica
   compatível" vs "Fit baixo — considere revisar")

## 1. Carregando o modelo treinado

Diferente dos notebooks anteriores (que baixavam o BERTimbau original da
Hugging Face), aqui carregamos a versão que **nós treinamos** — salva no
Drive ao final do Notebook 04, já com os pesos ajustados para prever
fit_percentual.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from transformers import AutoTokenizer, AutoModelForSequenceClassification
import torch

CAMINHO_MODELO = '/content/drive/MyDrive/jobmatch-ai/models/bertimbau_fit_v1'

tokenizer = AutoTokenizer.from_pretrained(CAMINHO_MODELO)
modelo = AutoModelForSequenceClassification.from_pretrained(CAMINHO_MODELO)

# Move para GPU se disponível
dispositivo = "cuda" if torch.cuda.is_available() else "cpu"
modelo.to(dispositivo)
modelo.eval()  # modo de avaliação (desliga dropout e outras camadas exclusivas de treino)

print(f"Modelo carregado com sucesso, rodando em: {dispositivo}")

Mounted at /content/drive


Loading weights:   0%|          | 0/201 [00:01<?, ?it/s]

Modelo carregado com sucesso, rodando em: cpu


## 2. Construindo a função prever_fit()

Esta é a função central do sistema: recebe o texto de uma vaga (string) e
retorna o fit_percentual estimado, junto com uma interpretação legível.

Passos internos da função:
1. Tokeniza o texto de entrada (mesmo processo do Notebook 02)
2. Passa pelo modelo em modo de inferência (sem calcular gradientes, mais
   rápido e economiza memória)
3. Desnormaliza a saída (que o modelo produz na escala 0-1) de volta para 0-100
4. Classifica o resultado numa categoria interpretável

In [ ]:
def prever_fit(texto_vaga, retornar_detalhes=True):
    """
    Recebe o texto de uma vaga e retorna o fit_percentual estimado pelo
    modelo fine-tuned, com interpretação.
    """
    # Tokenizar (mesmo max_length usado no treinamento)
    inputs = tokenizer(
        texto_vaga,
        padding='max_length',
        truncation=True,
        max_length=256,
        return_tensors='pt'
    ).to(dispositivo)

    # Inferência sem calcular gradientes (mais rápido, não estamos treinando)
    with torch.no_grad():
        saida = modelo(**inputs)
        predicao_normalizada = saida.logits.item()

    fit_percentual = predicao_normalizada * 100
    fit_percentual = max(0, min(100, fit_percentual))  # garante que fica entre 0-100

    # Interpretação legível
    if fit_percentual >= 70:
        interpretacao = "Fit alto — forte alinhamento com perfil de Ciência de Dados"
    elif fit_percentual >= 45:
        interpretacao = "Fit médio — área adjacente ou parcialmente alinhada"
    elif fit_percentual >= 20:
        interpretacao = "Fit baixo — pouca aderência ao perfil técnico"
    else:
        interpretacao = "Fit muito baixo — provavelmente fora da área de interesse"

    if retornar_detalhes:
        return {
            "fit_percentual": round(fit_percentual, 1),
            "interpretacao": interpretacao
        }
    return round(fit_percentual, 1)


print("Função prever_fit() criada com sucesso!")

Função prever_fit() criada com sucesso!


## 3. Testando a função com vagas inéditas

Testamos `prever_fit()` com vagas que **não estavam no dataset de treino nem
de validação** — é o teste mais honesto do sistema, simulando o uso real:

In [ ]:
# Vaga 1: Cientista de Dados Júnior (esperado: fit alto)
vaga_teste_1 = """
Estamos em busca de um Cientista de Dados Júnior para integrar nosso time de
Data Science. Responsabilidades incluem desenvolvimento de modelos preditivos,
análise exploratória de dados, criação de pipelines de Machine Learning e
apresentação de insights para áreas de negócio. Requisitos: Python, SQL,
conhecimento em bibliotecas como Pandas, Scikit-learn, e noções de estatística.
Diferencial: experiência com Power BI e Cloud (AWS/GCP).
"""

# Vaga 2: Auxiliar Administrativo (esperado: fit baixo)
vaga_teste_2 = """
Empresa contrata Auxiliar Administrativo para atuar no setor financeiro,
com responsabilidades de organização de documentos, atendimento a clientes,
lançamento de notas fiscais e apoio geral ao departamento. Requisitos:
Ensino médio completo, conhecimento em pacote Office, boa comunicação.
"""

# Vaga 3: Analista de BI Júnior (esperado: fit médio)
vaga_teste_3 = """
Buscamos Analista de BI Júnior para construção de dashboards, escrita de
consultas SQL, e apoio na modelagem de dados no Power BI. Desejável
conhecimento básico em Python para automação de relatórios.
"""

for i, vaga in enumerate([vaga_teste_1, vaga_teste_2, vaga_teste_3], 1):
    resultado = prever_fit(vaga)
    print(f"=== Vaga teste {i} ===")
    print(f"Fit previsto: {resultado['fit_percentual']}")
    print(f"Interpretação: {resultado['interpretacao']}")
    print()

=== Vaga teste 1 ===
Fit previsto: 73.7
Interpretação: Fit alto — forte alinhamento com perfil de Ciência de Dados

=== Vaga teste 2 ===
Fit previsto: 8.8
Interpretação: Fit muito baixo — provavelmente fora da área de interesse

=== Vaga teste 3 ===
Fit previsto: 49.8
Interpretação: Fit médio — área adjacente ou parcialmente alinhada



## Interpretação dos testes

O modelo classificou corretamente as três vagas de teste dentro das faixas
esperadas, mesmo sendo textos completamente inéditos (não presentes no
treino ou validação):

- Cientista de Dados Júnior: 73.7 (fit alto) ✅
- Auxiliar Administrativo: 8.8 (fit muito baixo) ✅
- Analista de BI Júnior: 49.8 (fit médio) ✅

Esse resultado demonstra que o modelo generalizou o padrão aprendido durante
o fine-tuning para vagas novas, validando o pipeline completo do projeto —
da coleta de dados até um sistema funcional de predição.

## 4. Interface simplificada para uso interativo

Para facilitar testes futuros (e servir de base para o futuro deploy via
FastAPI/Streamlit), criamos uma versão da função com saída formatada, pronta
para colar qualquer vaga e obter o resultado rapidamente.

In [ ]:
def analisar_vaga(texto_vaga):
    resultado = prever_fit(texto_vaga)
    print("=" * 50)
    print(f"FIT ESTIMADO: {resultado['fit_percentual']}/100")
    print(f"{resultado['interpretacao']}")
    print("=" * 50)

# Exemplo de uso — cole qualquer vaga aqui:
analisar_vaga("""
Cole aqui o texto de uma vaga real para testar
""")

FIT ESTIMADO: 30.5/100
Fit baixo — pouca aderência ao perfil técnico


## Conclusões — Sistema de Recomendação (Notebook 05)

- Construída a função `prever_fit()`, que recebe o texto de uma vaga e
  retorna o fit_percentual estimado pelo modelo fine-tuned, com interpretação
  legível em 4 categorias (muito baixo / baixo / médio / alto)

- Validação com 3 vagas inéditas (não presentes no treino/validação) confirmou
  que o modelo generaliza corretamente: Cientista de Dados (73.7 — fit alto),
  Auxiliar Administrativo (8.8 — fit muito baixo), Analista de BI (49.8 —
  fit médio)

- Esta função é a base para os próximos passos do projeto:
  - Deploy via FastAPI (endpoint /prever-fit)
  - Interface simples via Streamlit para uso não-técnico
  - Possível extensão futura: retornar não só o score, mas também os termos
    do texto que mais influenciaram a predição (interpretabilidade via
    attention weights)

- Limitações já